In [ ]:
import pandas as pd
import re
import nltk

# Download NLP resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Import NLP tools
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Initialize tools
stop_words = set(stopwords.words('english'))

# Keep "not" for sentiment meaning
stop_words.discard('not')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\abc.zip.
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]  

## Load Dataset

In [ ]:
df = pd.read_csv("../data/IMDB Dataset.csv")
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [18]:
## Data Overview
print(df.shape)
print(df['sentiment'].value_counts())

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Text Cleaning
- Convert to lowercase  
- Remove HTML tags  
- Remove punctuation  

In [19]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

## Tokenization & Stopwords Removal
Split text into words and remove common words.

In [20]:
def process_text(text):
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

## Lemmatization
Convert words to their base form.

In [21]:
def lemmatize(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

## Apply Preprocessing
Combine all steps into final cleaned text.

In [22]:
df['clean_review'] = df['review'].apply(clean_text)
df['tokens'] = df['clean_review'].apply(process_text)
df['tokens'] = df['tokens'].apply(lemmatize)

df['clean_review'] = df['tokens'].apply(lambda x: " ".join(x))

LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - 'C:\\Users\\Mohamed/nltk_data'
    - 'd:\\information\\college\\3rd year 1st term\\Material\\ML\\ml\\.venv\\nltk_data'
    - 'd:\\information\\college\\3rd year 1st term\\Material\\ML\\ml\\.venv\\share\\nltk_data'
    - 'd:\\information\\college\\3rd year 1st term\\Material\\ML\\ml\\.venv\\lib\\nltk_data'
    - 'C:\\Users\\Mohamed\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


## Label Encoding
Convert sentiment to numerical values.

In [ ]:
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:

print(df[['review','clean_review']].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


## Data Quality Check

In [ ]:
# Check nulls
df.isnull().sum()

review          0
sentiment       0
clean_review    0
tokens          0
label           0
dtype: int64

In [ ]:
# Check labels
df[['sentiment', 'label']].head()
df['label'].value_counts()


label
1    25000
0    25000
Name: count, dtype: int64

In [ ]:
# Check text length
df['clean_review'].apply(len).describe()

count    50000.000000
mean       824.462580
std        635.185492
min         17.000000
25%        432.000000
50%        606.000000
75%       1004.000000
max       9190.000000
Name: clean_review, dtype: float64

In [ ]:
# Remove empty reviews 
df = df[df['clean_review'].str.strip() != ""]

## Save Cleaned Data


In [ ]:
df.to_csv("../data/cleaned_data.csv", index=False)